# Warm-Start Action Cap Validation V1

Purpose: follow `metric_source_reconciliation_v1` by rerunning the selected action-cap candidate with official qlib backtests that keep pre-window `LOOKBACK` context in the strategy frame. The account and benchmark window still start at the fold test boundary; only the observation history is warm-started.

Governance:
- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sequence_risk_action_cap_screen_v1` and `metric_source_reconciliation_v1`.
- **Role**: `diagnose`.
- **Target window**: warm-started `wf_2022` and `wf_2023_2024`; target is `+30pp` relative return and `<=15%` MaxDD on both.
- **Changed layer**: `model_selection`, `harness_metric_diagnostic`, `src_capability`.
- **External reference**: qlib official backtest workflow plus prior qlib/PPO/safe-RL/PBO references: https://github.com/microsoft/qlib, https://arxiv.org/abs/2009.11189, https://arxiv.org/abs/1707.06347, https://arxiv.org/abs/1712.06924, https://arxiv.org/abs/2006.04779, https://doi.org/10.21314/JCF.2016.322.
- **Hypothesis**: if the previous metric mismatch was only warmup benchmark alignment, then warm-started official qlib evaluation should remove the forced-flat first-60-row artifact and let the calendar relative gate reflect tradable behavior.
- **Acceptance criteria**: selected candidate clears both warm-started folds; baseline comparison shows whether the cap adds drawdown value beyond the BC actor itself; no promotion until fixed-split / `wf_2020` are covered.
- **Failure mode**: selected candidate only passes because of benchmark reinterpretation, cap does not add value, or warm-start official qlib still fails one fold.
- **Comparison-table row key**: `warm_start_action_cap_validation_v1`.

Historical duplicate check:
- Closest success case: `metric_source_reconciliation_v1` showed the selected candidate passes two folds under tradable-window relative metrics with MaxDD below 15%.
- Closest failure case: `sequence_risk_action_cap_screen_v1` blocked promotion because isolated official qlib relative metrics were inconsistent with `evaluate_split`.
- This run is not a duplicate because it reruns official qlib with warm-started observation context instead of only recomputing benchmark variants.


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.path_diagnostics import make_backtest_daily_frame
from src.soxl_quant.app.sequence_risk_objective import (
    DEFAULT_TREND_BREAK_SPECS,
    fold_local_sequence_risk_thresholds,
    score_sequence_risk_path,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.dataset import slice_with_lookback
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'warm_start_action_cap_validation_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_ACTION_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'sequence_risk_action_cap_screen_v1'
SOURCE_METRIC_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'metric_source_reconciliation_v1'
SOURCE_TARGET_CAP_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_target_cap_bc_screen_v1'
TARGETS_PATH = SOURCE_TARGET_CAP_CARD / 'baseline_no_sequence_cap_target_contract.csv'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

CAP_RECIPES = [
    {'candidate': 'baseline_no_action_cap', 'target_cap': None, 'min_feature_hits': 0, 'description': 'BC baseline without action cap.'},
    {'candidate': 'seq_cap10_two_of_three', 'target_cap': 0.10, 'min_feature_hits': 2, 'description': 'Selected 10% cap when at least two trend-break features fire.'},
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)
print('lookback', LOOKBACK)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/warm_start_action_cap_validation_v1
lookback 60


In [2]:
profile = load_profile(PROFILE_NAME)
source_action_selection = json.loads((SOURCE_ACTION_CARD / 'selection.json').read_text(encoding='utf-8'))
source_metric_selection = json.loads((SOURCE_METRIC_CARD / 'selection.json').read_text(encoding='utf-8'))
source_action_metrics = pd.read_csv(SOURCE_ACTION_CARD / 'action_cap_metrics.csv')
source_metric_recon = pd.read_csv(SOURCE_METRIC_CARD / 'metric_reconciliation.csv')
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH).sort_index()
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start, train_end = map(pd.Timestamp, spec['train'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'source_action_decision': source_action_selection.get('decision'),
    'source_metric_decision': source_metric_selection.get('decision'),
    'lookback': LOOKBACK,
    'coverage': coverage.to_dict('records'),
})
coverage


,fold,target_fold,train_target_rows,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2022,wf_2022_train,253,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,wf_2023_2024,wf_2023_2024_train,756,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
def overlay_config_for_recipe(recipe: dict[str, Any], thresholds: dict[str, dict[str, Any]]) -> dict[str, Any] | None:
    if recipe['target_cap'] is None:
        return None
    return {
        'type': 'sequence_risk_action_cap',
        'target_cap': float(recipe['target_cap']),
        'sequence_risk_min_feature_hits': int(recipe['min_feature_hits']),
        'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
        'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
        'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
    }


def official_metrics(bt: dict[str, Any], price_series: pd.Series) -> dict[str, float | str]:
    effective_start, effective_end = bt.get('effective_test_seg', ('', ''))
    prices = price_series.loc[str(effective_start):str(effective_end)].dropna().astype(float)
    bnh = (float(prices.iloc[-1]) / float(prices.iloc[0]) - 1.0) * 100.0 if len(prices) >= 2 and float(prices.iloc[0]) else float('nan')
    ret = float(bt.get('return', float('nan')))
    pos_hist = np.asarray(bt.get('position_history', []), dtype=float)
    return {
        'return': ret,
        'sharpe': float(bt.get('sharpe', float('nan'))),
        'max_dd': float(bt.get('max_dd', float('nan'))),
        'buy_and_hold_return': bnh,
        'relative_return': ret - bnh if math.isfinite(ret) and math.isfinite(bnh) else float('nan'),
        'avg_pos': float(np.mean(pos_hist)) if pos_hist.size else 0.0,
        'stress_avg_pos': float(np.mean(pos_hist[pos_hist > 0.0])) if pos_hist.size and np.any(pos_hist > 0.0) else 0.0,
        'effective_start': str(effective_start),
        'effective_end': str(effective_end),
    }


def target_gate(metrics: dict[str, Any]) -> bool:
    return bool(float(metrics.get('relative_return', float('-inf'))) >= TARGET_RELATIVE_RETURN and float(metrics.get('max_dd', float('inf'))) <= TARGET_MAX_DD)


def trace_metrics(trace: list[dict[str, Any]]) -> dict[str, float | int | str]:
    if not trace:
        return {
            'trace_rows': 0,
            'first_trace_date': '',
            'overlay_ready_days': 0,
            'target_cap_changed_days': 0,
            'order_days': 0,
            'blocked_by_trade_lock_days': 0,
            'blocked_by_min_rebalance_days': 0,
        }
    frame_trace = pd.DataFrame(trace)
    changed = pd.to_numeric(frame_trace.get('target_after_overlay', 0.0), errors='coerce') < pd.to_numeric(frame_trace.get('target_before_overlay', 0.0), errors='coerce') - 1e-9
    return {
        'trace_rows': int(len(frame_trace)),
        'first_trace_date': str(pd.to_datetime(frame_trace['date']).min().date()) if 'date' in frame_trace else '',
        'overlay_ready_days': int(frame_trace.get('overlay_gate_ready', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
        'target_cap_changed_days': int(changed.fillna(False).sum()),
        'order_days': int(pd.to_numeric(frame_trace.get('order_count', pd.Series(0, index=frame_trace.index)), errors='coerce').fillna(0).gt(0).sum()),
        'blocked_by_trade_lock_days': int(frame_trace.get('blocked_by_trade_lock', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
        'blocked_by_min_rebalance_days': int(frame_trace.get('blocked_by_min_rebalance', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
    }


def prefixed(prefix: str, metrics: dict[str, Any]) -> dict[str, Any]:
    return {f'{prefix}_{key}': value for key, value in metrics.items()}


In [4]:
init_qlib(QLIB_DATA_PATH)
feature_cols = feature_columns_for_packs(profile.feature.packs, profile.name)
feature_frame_for_daily = frame[['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63']].copy()

fold_rows = []
slice_rows = []
threshold_rows = []
daily_paths: dict[tuple[str, str], pd.DataFrame] = {}
trace_frames = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] train={spec['train']} test={spec['test']}")
    set_seed(SEED)
    train_start, train_end = map(pd.Timestamp, spec['train'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    if fold_targets.empty:
        fold_rows.append({'fold': fold_name, 'candidate': 'all', 'status': 'skipped_no_targets'})
        continue
    thresholds = fold_local_sequence_risk_thresholds(frame, fold_targets['datetime'], specs=DEFAULT_TREND_BREAK_SPECS)
    threshold_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'thresholds_json': json.dumps(json_safe(thresholds), sort_keys=True),
        'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
        'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
        'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
    })

    fit = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=TRAIN_CONFIG,
        sample_weight_config=None,
        sample_size=0,
    )
    dataset_for_bc = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
    slice_metrics = evaluate_bc_actor_slice_fit(
        fit.actor,
        fit.obs,
        fit.actions,
        dataset_for_bc.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    tagged_slices = slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    slice_rows.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    full_df = pd.concat([
        segment_frame(dataset, 'train'),
        segment_frame(dataset, 'valid'),
        segment_frame(dataset, 'test'),
    ]).sort_index()
    price_full = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    warm_df = slice_with_lookback(full_df, spec['test'][0], spec['test'][1], lookback=LOOKBACK)
    warm_price = slice_with_lookback(price_full, spec['test'][0], spec['test'][1], lookback=LOOKBACK)
    warmup_context_rows = int((pd.to_datetime(warm_df.index) < pd.Timestamp(spec['test'][0])).sum())

    for recipe in CAP_RECIPES:
        candidate = str(recipe['candidate'])
        overlay = overlay_config_for_recipe(recipe, thresholds)
        bt = backtest_profile(
            fit.actor,
            warm_df,
            warm_price,
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_cols,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=spec['test'],
            overlay_config=overlay,
            trace_decisions=True,
        )
        metrics = official_metrics(bt, warm_price)
        daily = make_backtest_daily_frame(bt, label=f'{candidate}_{fold_name}', feature_frame=feature_frame_for_daily)
        daily_paths[(candidate, fold_name)] = daily
        seq_score = score_sequence_risk_path(daily, thresholds=thresholds, mode='two_of_three')
        trace = pd.DataFrame(bt.get('decision_trace', []))
        if not trace.empty:
            trace.insert(0, 'candidate', candidate)
            trace.insert(0, 'fold', fold_name)
            trace_frames.append(trace)
        isolated = source_action_metrics[source_action_metrics['candidate'].eq(candidate) & source_action_metrics['fold'].eq(fold_name)]
        fold_rows.append({
            'candidate': candidate,
            'fold': fold_name,
            'status': 'evaluated',
            'target_cap': np.nan if recipe['target_cap'] is None else float(recipe['target_cap']),
            'min_feature_hits': int(recipe['min_feature_hits']),
            'description': str(recipe['description']),
            'bc_after_accuracy': float(fit.metrics['after_accuracy']),
            'bc_after_loss': float(fit.metrics['after_loss']),
            'warmup_context_rows': warmup_context_rows,
            'target_gate': target_gate(metrics),
            **metrics,
            'isolated_return': float(isolated['return'].iloc[0]) if not isolated.empty else np.nan,
            'isolated_max_dd': float(isolated['max_dd'].iloc[0]) if not isolated.empty else np.nan,
            'isolated_relative_return': float(isolated['relative_return'].iloc[0]) if not isolated.empty else np.nan,
            **prefixed('trace', trace_metrics(bt.get('decision_trace', []))),
            **prefixed('sequence', seq_score),
        })

thresholds_df = pd.DataFrame(threshold_rows)
metrics_df = pd.DataFrame(fold_rows)
slices_df = pd.concat(slice_rows, ignore_index=True, sort=False) if slice_rows else pd.DataFrame()
traces_df = pd.concat(trace_frames, ignore_index=True, sort=False) if trace_frames else pd.DataFrame()
thresholds_df.to_csv(CARD / 'fold_sequence_thresholds.csv', index=False)
metrics_df.to_csv(CARD / 'warm_start_action_cap_metrics.csv', index=False)
slices_df.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
traces_df.to_csv(CARD / 'warm_start_decision_trace.csv', index=False)

for (candidate, fold_name), daily in daily_paths.items():
    if candidate in {'baseline_no_action_cap', 'seq_cap10_two_of_three'}:
        daily.to_csv(CARD / f'{candidate}_{fold_name}_daily_path.csv', index=False)

metrics_df


[61684:MainThread](2026-05-26 11:59:01,373) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61684:MainThread](2026-05-26 11:59:02,022) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61684:MainThread](2026-05-26 11:59:02,023) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] train=('2010-03-11', '2020-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,status,target_cap,min_feature_hits,description,bc_after_accuracy,bc_after_loss,warmup_context_rows,target_gate,...,sequence_sequence_position_lift,sequence_sequence_max_dd_pct,sequence_drawdown_breach_rows,sequence_sequence_breach_rows,sequence_sequence_breach_capture,sequence_sequence_hit_count_mean,sequence_sequence_exposure_loss,sequence_sequence_drawdown_loss,sequence_sequence_breach_capture_loss,sequence_sequence_path_penalty
0,baseline_no_action_cap,wf_2022,evaluated,NaN,0,BC baseline without action cap.,0.833992,1.073453,60,False,...,0.092913,23.788496,84,84,1.0,1.676,0.578825,0.5859,0.578825,0.581655
1,seq_cap10_two_of_three,wf_2022,evaluated,0.1,2,Selected 10% cap when at least two trend-break...,0.833992,1.073453,60,True,...,0.014710,13.280734,0,0,0.0,1.676,0.000000,0.0000,0.000000,0.000000
2,baseline_no_action_cap,wf_2023_2024,evaluated,NaN,0,BC baseline without action cap.,0.768212,1.146839,60,False,...,-0.160573,0.000000,82,0,0.0,0.000,0.000000,0.0000,0.000000,0.000000
3,seq_cap10_two_of_three,wf_2023_2024,evaluated,0.1,2,Selected 10% cap when at least two trend-break...,0.768212,1.146839,60,False,...,-0.160573,0.000000,82,0,0.0,0.000,0.000000,0.0000,0.000000,0.000000


In [5]:
summary_rows = []
for candidate, group in metrics_df.groupby('candidate', sort=False):
    group = group.sort_values('fold')
    summary_rows.append({
        'candidate': candidate,
        'min_relative_return': float(group['relative_return'].min()),
        'max_max_dd': float(group['max_dd'].max()),
        'all_target_gate': bool(group['target_gate'].all()),
        'mean_avg_pos': float(group['avg_pos'].mean()),
        'target_cap_changed_days_total': int(group['trace_target_cap_changed_days'].sum()),
        'warmup_context_rows_min': int(group['warmup_context_rows'].min()),
        'first_trace_dates': ','.join(group['trace_first_trace_date'].astype(str).tolist()),
        'wf_2022_relative_return': float(group[group['fold'].eq('wf_2022')]['relative_return'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2022_max_dd': float(group[group['fold'].eq('wf_2022')]['max_dd'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2023_2024_relative_return': float(group[group['fold'].eq('wf_2023_2024')]['relative_return'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
        'wf_2023_2024_max_dd': float(group[group['fold'].eq('wf_2023_2024')]['max_dd'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
    })

candidate_summary = pd.DataFrame(summary_rows).sort_values(
    ['all_target_gate', 'max_max_dd', 'target_cap_changed_days_total'],
    ascending=[False, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / 'warm_start_candidate_summary.csv', index=False)
selected = candidate_summary[candidate_summary['candidate'].eq('seq_cap10_two_of_three')].iloc[0].to_dict()
baseline = candidate_summary[candidate_summary['candidate'].eq('baseline_no_action_cap')].iloc[0].to_dict()

candidate_summary


,candidate,min_relative_return,max_max_dd,all_target_gate,mean_avg_pos,target_cap_changed_days_total,warmup_context_rows_min,first_trace_dates,wf_2022_relative_return,wf_2022_max_dd,wf_2023_2024_relative_return,wf_2023_2024_max_dd
0,seq_cap10_two_of_three,3.126126,20.232446,False,0.118220,106,60,"2022-01-03,2024-01-02",77.300753,13.280734,3.126126,20.232446
1,baseline_no_action_cap,3.126126,23.788496,False,0.147573,0,60,"2022-01-03,2024-01-02",64.760981,23.788496,3.126126,20.232446


In [6]:
selected_passes_two_folds = bool(selected['all_target_gate'])
baseline_passes_two_folds = bool(baseline['all_target_gate'])
cap_adds_drawdown_value = float(selected['max_max_dd']) < float(baseline['max_max_dd']) - 1e-9
if selected_passes_two_folds and cap_adds_drawdown_value:
    decision = 'warm_start_action_cap_two_fold_pass_no_promotion_pending_fixed_split_wf2020'
    next_mechanism = 'generate_or_reuse_nonleaking_wf2020_target_then_fixed_split_warm_start_validation'
elif selected_passes_two_folds:
    decision = 'warm_start_bc_actor_two_fold_pass_cap_value_unclear_no_promotion'
    next_mechanism = 'compare_cap_vs_baseline_on_fixed_split_wf2020_before_any_runtime_overlay'
else:
    decision = 'warm_start_action_cap_failed_no_promotion'
    next_mechanism = 'return_to_training_objective_or_checkpoint_selection_not_threshold_search'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': ['sequence_risk_action_cap_screen_v1', 'metric_source_reconciliation_v1'],
    'profile': PROFILE_NAME,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'decision': decision,
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'selected_candidate': 'seq_cap10_two_of_three',
    'selected_all_target_gate': selected_passes_two_folds,
    'baseline_all_target_gate': baseline_passes_two_folds,
    'cap_adds_drawdown_value': cap_adds_drawdown_value,
    'selected_min_relative_return': float(selected['min_relative_return']),
    'selected_max_max_dd': float(selected['max_max_dd']),
    'baseline_min_relative_return': float(baseline['min_relative_return']),
    'baseline_max_max_dd': float(baseline['max_max_dd']),
    'selected_target_cap_changed_days_total': int(selected['target_cap_changed_days_total']),
    'next_mechanism': next_mechanism,
    'promotion_blocker': 'This card validates only two warm-started folds. Promotion still requires fixed-split/test_proxy plus non-leaking wf_2020 validation and registry review.',
    'sidecars': [
        'coverage.csv', 'coverage.json', 'fold_sequence_thresholds.csv', 'bc_slice_metrics.csv',
        'warm_start_action_cap_metrics.csv', 'warm_start_candidate_summary.csv',
        'warm_start_decision_trace.csv', 'selection.json', 'summary.json',
        'baseline_no_action_cap_wf_2022_daily_path.csv', 'baseline_no_action_cap_wf_2023_2024_daily_path.csv',
        'seq_cap10_two_of_three_wf_2022_daily_path.csv', 'seq_cap10_two_of_three_wf_2023_2024_daily_path.csv',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1712.06924',
        'https://arxiv.org/abs/2006.04779',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    **selection,
    'source_action_decision': source_action_selection.get('decision'),
    'source_metric_decision': source_metric_selection.get('decision'),
    'lookback': LOOKBACK,
    'fold_windows': WALK_FORWARD_WINDOWS,
    'coverage': coverage.to_dict('records'),
    'candidate_summary': candidate_summary.to_dict('records'),
    'metrics': metrics_df.to_dict('records'),
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
selection


{'experiment': 'warm_start_action_cap_validation_v1',
 'created_at': '2026-05-26T03:59:37.559806+00:00',
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['sequence_risk_action_cap_screen_v1',
  'metric_source_reconciliation_v1'],
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'decision': 'warm_start_action_cap_failed_no_promotion',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'selected_candidate': 'seq_cap10_two_of_three',
 'selected_all_target_gate': False,
 'baseline_all_target_gate': False,
 'cap_adds_drawdown_value': True,
 'selected_min_relative_return': 3.1261263775804626,
 'selected_max_max_dd': 20.23244637943855,
 'baseline_min_relative_return': 3.1261263775804626,
 'baseline_max_max_dd': 23.788496458260497,
 'selected_target_cap_changed_days_total': 106,
 'next_mechanism': 'return_to_training_objective_or_checkpoint_se

In [7]:
selected_rows = metrics_df[metrics_df['candidate'].eq('seq_cap10_two_of_three')].sort_values('fold')
baseline_rows = metrics_df[metrics_df['candidate'].eq('baseline_no_action_cap')].sort_values('fold')
lines = []
for row in selected_rows.to_dict('records'):
    base = baseline_rows[baseline_rows['fold'].eq(row['fold'])].iloc[0].to_dict()
    lines.append(
        f"- {row['fold']}: selected rel `{row['relative_return']:.2f}pp`, MaxDD `{row['max_dd']:.2f}%`, "
        f"target changes `{int(row['trace_target_cap_changed_days'])}`; baseline rel `{base['relative_return']:.2f}pp`, MaxDD `{base['max_dd']:.2f}%`."
    )

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`. Promotion remains blocked.

Warm-start setup: feature and price frames include up to `LOOKBACK={LOOKBACK}` rows before the fold start, while qlib account and benchmark still start at each fold's test boundary. This removes the forced-flat first-60-row artifact from the isolated source card.

Selected candidate `{selection['selected_candidate']}`:
{chr(10).join(lines)}

Interpretation: selected two-fold gate = `{selection['selected_all_target_gate']}`, baseline two-fold gate = `{selection['baseline_all_target_gate']}`, cap adds drawdown value = `{selection['cap_adds_drawdown_value']}`. This is still diagnostic evidence only because `wf_2020` and fixed-split/test_proxy are not covered by this fold-local target contract.

Next action: `{selection['next_mechanism']}`. Do not promote or retain an actor from this card.

Artifacts: `warm_start_action_cap_metrics.csv`, `warm_start_candidate_summary.csv`, `warm_start_decision_trace.csv`, fold daily paths, `selection.json`, `summary.json`.
""".strip()

display(Markdown(conclusion))


## Conclusion

Decision: `warm_start_action_cap_failed_no_promotion`. Promotion remains blocked.

Warm-start setup: feature and price frames include up to `LOOKBACK=60` rows before the fold start, while qlib account and benchmark still start at each fold's test boundary. This removes the forced-flat first-60-row artifact from the isolated source card.

Selected candidate `seq_cap10_two_of_three`:
- wf_2022: selected rel `77.30pp`, MaxDD `13.28%`, target changes `106`; baseline rel `64.76pp`, MaxDD `23.79%`.
- wf_2023_2024: selected rel `3.13pp`, MaxDD `20.23%`, target changes `0`; baseline rel `3.13pp`, MaxDD `20.23%`.

Interpretation: selected two-fold gate = `False`, baseline two-fold gate = `False`, cap adds drawdown value = `True`. This is still diagnostic evidence only because `wf_2020` and fixed-split/test_proxy are not covered by this fold-local target contract.

Next action: `return_to_training_objective_or_checkpoint_selection_not_threshold_search`. Do not promote or retain an actor from this card.

Artifacts: `warm_start_action_cap_metrics.csv`, `warm_start_candidate_summary.csv`, `warm_start_decision_trace.csv`, fold daily paths, `selection.json`, `summary.json`.